Task : Build a Chatbot using Hugging Face Transformers

Step-1:Install the requirements

In [3]:
!pip install -q -U "transformers>=4.37.0" accelerate

Step-2:Importing the packages

In [10]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

Step-3:Loading the model

Here we are using Qwen which is a lightweight instruction-tuned language model from Alibaba under the Qwen (Tongyi) series.

It has 0.5 billion parameter (0.5B) transformer model,designed for instruction-following tasks (chat, Q&A, basic reasoning) and it is part of the newer Qwen2.5 family

In [6]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
print("Model loaded successfully!\n")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model loaded successfully!



Step-4: Defining response function 

the response function Takes user input + history and Returns model response + updated history.

Here,the input is take, then adds it to the history, generates a response and also adds that to the history

In [ ]:
def generate_response(user_input, chat_history):
    #Add user message to the history
    chat_history.append({"role": "user", "content": user_input})
    #Formats the conversation using chat template
    input_text = tokenizer.apply_chat_template(
        chat_history,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    #it generates the response
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.7,
        top_p=0.9,
        do_sample=True
    )
    #decodes only the new tokens
    response = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[-1]:],
        skip_special_tokens=True
    )
    #Adds the bot response to history
    chat_history.append({"role": "assistant", "content": response})
    return response, chat_history

Step-5:defining loop for chatbot

this fucntion keeps the bot running until we dont type exit or quit.

In [8]:
def run_chatbot():
    print("Chatbot: Hello! I am your AI assistant. How can I help you today?\n")

    chat_history = []

    while True:
        user_input = input("You: ")

        # Exit condition
        if user_input.lower() in ["exit", "quit"]:
            print("Chatbot: Goodbye! Have a great day 👋")
            break

        # Generate response
        reply, chat_history = generate_response(user_input, chat_history)

        print(f"Chatbot: {reply}\n")


In [12]:
run_chatbot()

Chatbot: Hello! I am your AI assistant. How can I help you today?



You:  hello


Chatbot: Hello! How can I assist you today?



You:  What is Artificial Intelligence?


Chatbot: Artificial Intelligence (AI) refers to the simulation of human intelligence in machines that are programmed to think and act like humans. It involves developing algorithms and models that allow computers to perform tasks that typically require human intelligence, such as perception, learning, reasoning, problem-solving, decision-making, creativity, and language understanding.

Key aspects of AI include:
1. Machine Learning: This involves training algorithms on large datasets using supervised and unsupervised methods to learn patterns and improve their performance over time.
2. Deep Learning: A subset of machine learning that uses artificial neural networks with many layers to perform complex tasks.
3. Natural Language Processing (NLP): The ability of machines to understand and interpret human language, including speech, written text, and visual data



...

Chatbot: Python was created by Guido van Rossum, a Dutch software engineer who lived from 1956 to 1991. He developed Python in his free time while working at EurekLab in Amsterdam. Guido's motivation for creating Python was mainly due to its simplicity and ease of use, which made it easier for others to learn and use the language compared to more complex languages of the time. Python has since become one of the most popular programming languages globally, with over 20 million active users worldwide.



You:  thank you


Chatbot: You're welcome! If you have any other questions, feel free to ask.



You:  exit


Chatbot: Goodbye! Have a great day 👋


Chatbot Workflow:

User Input → Formatting → Tokenization → Model Generation → Response Decoding → Display Output → Repeat Until Exit

How the code works:
1.user enter text/question 

2.The text is converted to tokens 

3.model processes the input and generates a response

4.All the inputs and responses are stored 

5.type exit or quit to stop the converstion
